In [1]:
%%capture
%pip install transformers peft torch

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model_id = "HuggingFaceTB/SmolLM3-3B-Base"
adapter_path = "paul-stansifer/qw3-smollm3-3b-1x2e-4"
out_dir = "merged-model"

/home/paul/src/qwantzle-search/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load base model on CPU (force CPU to avoid GPU OOM).
# Use torch_dtype=torch.float32 or float16 depending on base availability.
base = AutoModelForCausalLM.from_pretrained(base_model_id, device_map="cpu", torch_dtype=torch.float32, low_cpu_mem_usage=True)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:20<00:00, 10.23s/it]


In [4]:
# Wrap with the adapter
model_with_adapter = PeftModel.from_pretrained(base, adapter_path, device_map="cpu")

In [ ]:
# Merge the LoRA weights into the base model and unload adapter
# NOTE: API name may be `merge_and_unload()` for LORA; if not present, use peft docs' recommended merge utilities.
# This call will fold adapter weights into the base model parameters.
model_with_adapter.merge_and_unload()

# Save the merged model (use safetensors for speed/compactness)
model_with_adapter.save_pretrained(out_dir, safe_serialization=True)

# Save tokenizer (important for later)
tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_fast=False)
tokenizer.save_pretrained(out_dir)

print("Merged model saved to", out_dir)

Merged model saved to merged-model


: 